# 02 — Préprocessing

Chaque choix est justifié. Tout ce qui dépend des données (moyennes, catégories) est appris **uniquement sur le train**.

In [1]:
from IPython.display import display

from obesity.data import prepare
from obesity.preprocessing import build_preprocessor, make_split, xy

df = prepare()
print(df.shape)

brut=(2111, 17) nettoye=(2087, 17) hash=8e8c54b782c6ddec
(2087, 17)


## 1. Nettoyage déterministe (avant le split)
- **Doublons supprimés** : évite qu'un exemple identique soit à la fois en train et en test.
- **Ordinales arrondies à l'entier** (FCVC, NCP, CH2O, FAF, TUE) : les valeurs décimales sont des artefacts SMOTE, ces variables sont des échelles entières.
- **`CALC = Always` fusionnée dans `Frequently`** : modalité quasi absente (≈1 ligne), impossible à apprendre et à stratifier.

## 2. Split unique 70/30 stratifié
Un seul split (`random_state=42`), indices sauvegardés dans `data/processed/split_indices.json` et réutilisés par **tous** les pipelines : les comparaisons A/B/C portent donc sur les mêmes individus.

In [2]:
tr, te = make_split(df)
print(len(tr), len(te), 'recouvrement :', len(set(tr) & set(te)))
display(df.loc[tr,'NObeyesdad'].value_counts(normalize=True).round(3).to_frame('train').join(df.loc[te,'NObeyesdad'].value_counts(normalize=True).round(3).to_frame('test')))

1460 627 recouvrement : 0


,train,test
NObeyesdad,,
Obesity_Type_I,0.168,0.169
Obesity_Type_III,0.155,0.155
Obesity_Type_II,0.142,0.142
Overweight_Level_II,0.139,0.139
Normal_Weight,0.135,0.136
Overweight_Level_I,0.132,0.132
Insufficient_Weight,0.128,0.128


## 3. ColumnTransformer
- **Binaires** (yes/no) → 0/1.
- **Ordinales textuelles** (CAEC, CALC) → `OrdinalEncoder` avec l'ordre explicite no < Sometimes < Frequently < Always (l'ordre est porteur d'information).
- **Nominales** (Gender, MTRANS) → one-hot avec `drop='first'` (évite la colinéarité parfaite pour la régression) et `handle_unknown='ignore'`.
- **Numériques** → `StandardScaler` (nécessaire pour le SVM, le MLP et pour comparer les coefficients).

Pipeline A : sans `Weight` ni `Height` (fuite). Pipeline B : `Height` incluse, `Weight` est la cible.

In [3]:
Xa, ya = xy(df.loc[tr], 'classification')
Xb, yb = xy(df.loc[tr], 'regression')
print('A :', list(Xa.columns))
print('B :', list(Xb.columns))
Xt = build_preprocessor('classification').fit_transform(Xa)
print(Xt.shape, 'NaN :', int(Xt.isna().sum().sum()))
display(Xt.head())

A : ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC', 'CAEC', 'CALC', 'Gender', 'MTRANS', 'Age', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
B : ['family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC', 'CAEC', 'CALC', 'Gender', 'MTRANS', 'Age', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE', 'Height']
(1460, 17) NaN : 0


,family_history_with_overweight,FAVC,SMOKE,SCC,CAEC,CALC,Gender_Male,MTRANS_Bike,MTRANS_Motorbike,MTRANS_Public_Transportation,MTRANS_Walking,Age,FCVC,NCP,CH2O,FAF,TUE
0,1,0,0,0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.516028,-0.775451,0.384267,-0.019988,-1.138343,0.485781
1,1,0,1,1,1.0,1.0,0.0,0.0,0.0,1.0,0.0,-0.516028,0.963656,0.384267,1.439166,2.262709,-1.010507
2,1,0,0,0,1.0,2.0,1.0,0.0,0.0,1.0,0.0,-0.201423,-0.775451,0.384267,-0.019988,1.129025,0.485781
3,0,0,0,0,1.0,2.0,1.0,0.0,0.0,0.0,1.0,0.427787,0.963656,0.384267,-0.019988,1.129025,-1.010507
4,0,0,0,0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,-0.358726,-0.775451,-2.109201,-0.019988,-1.138343,-1.010507
